In [16]:
import os
import pandas as pd
from dotenv import load_dotenv
load_dotenv()
api_key=os.getenv("GROQ_API_KEY")
from groq import Groq

client = Groq(api_key=api_key)

models = client.models.list()
print(models)

ModelListResponse(data=[Model(id='qwen/qwen3-32b', created=1748396646, object='model', owned_by='Alibaba Cloud', active=True, context_window=131072, public_apps=None, max_completion_tokens=40960), Model(id='openai/gpt-oss-safeguard-20b', created=1761708789, object='model', owned_by='OpenAI', active=True, context_window=131072, public_apps=None, max_completion_tokens=65536), Model(id='whisper-large-v3-turbo', created=1728413088, object='model', owned_by='OpenAI', active=True, context_window=448, public_apps=None, max_completion_tokens=448), Model(id='meta-llama/llama-prompt-guard-2-86m', created=1748632165, object='model', owned_by='Meta', active=True, context_window=512, public_apps=None, max_completion_tokens=512), Model(id='openai/gpt-oss-120b', created=1754408224, object='model', owned_by='OpenAI', active=True, context_window=131072, public_apps=None, max_completion_tokens=65536), Model(id='llama-3.1-8b-instant', created=1693721698, object='model', owned_by='Meta', active=True, cont

In [17]:
model_data=[]

for model in models.data:
    model_info = {
        "id": model.id,
        "object": getattr(model, "object", None),
        "created": getattr(model, "created", None),
        "owned_by": getattr(model, "owned_by", None),
        "active": getattr(model, "active", None),
        
        # Some APIs include extra fields (handle safely)
        "context_window": getattr(model, "context_window", None),
        "public_apps": getattr(model, "public_apps", None),
        "max_completion_tokens": getattr(model, "max_completion_tokens", None)
    }
    
    model_data.append(model_info)

In [18]:
df_models = pd.DataFrame(model_data)
df_models["created"] = pd.to_datetime(df_models["created"], unit="s", errors="coerce")

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
import pandas as pd
import time

URL = "https://console.groq.com/docs/rate-limits"

# Headless browser setup
options = Options()
options.add_argument("--headless")
options.add_argument("--disable-gpu")

driver = webdriver.Chrome(options=options)
driver.get(URL)

# Wait for page to fully load
time.sleep(5)

tables = driver.find_elements(By.TAG_NAME, "table")

all_dataframes = []

for table in tables:
    headers = [th.text for th in table.find_elements(By.TAG_NAME, "th")]
    
    rows = table.find_elements(By.TAG_NAME, "tr")[1:]
    
    data = []
    for row in rows:
        cols = [td.text for td in row.find_elements(By.TAG_NAME, "td")]
        if cols:
            data.append(cols)
    
    if data:
        df = pd.DataFrame(data, columns=headers if headers else None)
        all_dataframes.append(df)

driver.quit()

# Combine all tables
final_df = pd.concat(all_dataframes, ignore_index=True)

In [ ]:
final_df = final_df[final_df[0].notna()]
final_df = final_df.drop(columns=['HEADER', 'VALUE', 'NOTES'])

col_mapping = {
    0: "model",
    1: "RPM",
    2: "RPD",
    3: "TPM",
    4: "TPD",
    5: "ASH",
    6: "ASD"
}

final_df.rename(columns=col_mapping, inplace=True)
final_df

,0,1,2,3,4,5,6
0,canopylabs/orpheus-arabic-saudi,10,100,1.2K,3.6K,-,-
1,canopylabs/orpheus-v1-english,10,100,1.2K,3.6K,-,-
2,groq/compound,30,250,70K,-,-,-
3,groq/compound-mini,30,250,70K,-,-,-
4,llama-3.1-8b-instant,30,14.4K,6K,500K,-,-
5,llama-3.3-70b-versatile,30,1K,12K,100K,-,-
6,meta-llama/llama-4-scout-17b-16e-instruct,30,1K,30K,500K,-,-
7,meta-llama/llama-prompt-guard-2-22m,30,14.4K,15K,500K,-,-
8,meta-llama/llama-prompt-guard-2-86m,30,14.4K,15K,500K,-,-
9,openai/gpt-oss-120b,30,1K,8K,200K,-,-


In [27]:
df_Final=df_models.merge(final_df, left_on="id", right_on="model", how="inner")[["model","object", "created", "owned_by", "active", "context_window", "public_apps", "max_completion_tokens",  "RPM", "RPD", "TPM", "TPD", "ASH", "ASD"]]
df_Final

,model,object,created,owned_by,active,context_window,public_apps,max_completion_tokens,RPM,RPD,TPM,TPD,ASH,ASD
0,qwen/qwen3-32b,model,2025-05-28 01:44:06,Alibaba Cloud,True,131072,None,40960,60,1K,6K,500K,-,-
1,openai/gpt-oss-safeguard-20b,model,2025-10-29 03:33:09,OpenAI,True,131072,None,65536,30,1K,8K,200K,-,-
2,whisper-large-v3-turbo,model,2024-10-08 18:44:48,OpenAI,True,448,None,448,20,2K,-,-,7.2K,28.8K
3,meta-llama/llama-prompt-guard-2-86m,model,2025-05-30 19:09:25,Meta,True,512,None,512,30,14.4K,15K,500K,-,-
4,openai/gpt-oss-120b,model,2025-08-05 15:37:04,OpenAI,True,131072,None,65536,30,1K,8K,200K,-,-
5,llama-3.1-8b-instant,model,2023-09-03 06:14:58,Meta,True,131072,None,131072,30,14.4K,6K,500K,-,-
6,meta-llama/llama-4-scout-17b-16e-instruct,model,2025-04-05 17:40:24,Meta,True,131072,None,8192,30,1K,30K,500K,-,-
7,groq/compound-mini,model,2025-09-04 01:35:07,Groq,True,131072,None,8192,30,250,70K,-,-,-
8,canopylabs/orpheus-arabic-saudi,model,2025-12-16 23:07:19,Canopy Labs,True,4000,None,50000,10,100,1.2K,3.6K,-,-
9,canopylabs/orpheus-v1-english,model,2025-12-19 23:18:36,Canopy Labs,True,4000,None,50000,10,100,1.2K,3.6K,-,-


In [29]:
from langchain_groq import ChatGroq
from model_registry import get_models   # 👈 import from your file

GROQ_API_KEY = api_key


def load_llm(task="chat"):
    models = get_models(task)   # 👈 get models from registry

    for model in models:
        try:
            llm = ChatGroq(
                groq_api_key=GROQ_API_KEY,
                model_name=model,
                temperature=0
            )
            print(f"✅ Using model: {model}")
            return llm
        except Exception as e:
            print(f"❌ Failed: {model} -> {e}")

    raise ValueError("No working model found")


# 👇 USE IT
llm = load_llm(task="chat")

response = llm.invoke("Explain AI in simple terms")
print(response.content)

✅ Using model: llama-3.1-8b-instant
**What is AI?**

Artificial Intelligence (AI) is a way to make computers and machines think and act like humans. It's like giving a computer a brain, so it can learn, understand, and make decisions on its own.

**How does AI work?**

AI uses special algorithms (like recipes for computers) that help it:

1. **Learn**: AI can learn from data, like pictures, words, or numbers. It can recognize patterns and relationships between them.
2. **Understand**: AI can understand language, like what we say or write. It can also understand images, sounds, and other types of data.
3. **Make decisions**: AI can make decisions based on what it's learned and understood. It can choose the best option or take action.

**Types of AI**

There are two main types of AI:

1. **Narrow or Weak AI**: This type of AI is designed to perform a specific task, like playing chess or recognizing faces. It's like a specialized tool that's good at one thing.
2. **General or Strong AI**: